In [26]:
from tensorflow.keras.models import Model
from tensorflow.keras.applications import MobileNetV3Large
from tensorflow.keras.applications.mobilenet_v3 import preprocess_input 
from tensorflow.keras.layers import Dense,GlobalAveragePooling2D,Dropout
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import EarlyStopping,ReduceLROnPlateau
from tensorflow.keras.optimizers import Adam
from sklearn.metrics import classification_report,confusion_matrix
import clahe

In [27]:
training_path = "../Datasets/data/train/Medical/X-Ray/elbow/Train"
testing_path = "../Datasets/data/test/Medical/X-Ray/elbow/Test"
valid_path = "../Datasets/data/vaild/Medical/X-Ray/elbow/valid"

In [28]:
train_dataaug = ImageDataGenerator(
    rotation_range = 10,
    width_shift_range=0.05,
    height_shift_range=0.05,
    zoom_range=0.10,
    shear_range=0.05,
    horizontal_flip=True,
    preprocessing_function=lambda img: clahe.clahe_preprocess(img,preprocess_input)
)
testing_dataaug = ImageDataGenerator(preprocessing_function=lambda img : clahe.clahe_preprocess(img,preprocess_input))
valid_dataaug = ImageDataGenerator(preprocessing_function=lambda img : clahe.clahe_preprocess(img,preprocess_input))

In [29]:
train_set = train_dataaug.flow_from_directory(
    training_path,
    target_size=(224,224),
    batch_size=16,
    class_mode="binary"
)

valid_set = valid_dataaug.flow_from_directory(
    valid_path,
    target_size=(224,224),
    batch_size=16,
    class_mode="binary",
    shuffle=False
)

test_set = testing_dataaug.flow_from_directory(
    testing_path,
    target_size=(224,224),
    batch_size=16,
    class_mode="binary",
    shuffle=False
)

Found 8479 images belonging to 2 classes.
Found 1016 images belonging to 2 classes.
Found 1951 images belonging to 2 classes.


In [30]:
base_model = MobileNetV3Large(weights="imagenet",include_top=False,input_shape=(224,224,3),pooling="avg")

In [31]:
base_model.trainable = False

In [32]:
x = base_model.output
x = Dense(256,activation="relu")(x)
output = Dense(1,activation="sigmoid")(x)

In [33]:
model = Model(inputs = base_model.input,outputs = output)

In [ ]:
model.compile(loss="binary_crossentropy",optimizer=Adam(learning_rate=1e-3),metrics=['accuracy'])

In [35]:
class_weight = {0: 1.6,1: 1}

In [36]:
early_stop = EarlyStopping(
    monitor="val_loss",
    patience=4,
    restore_best_weights= True
)

In [37]:
model.fit(train_set,validation_data=valid_set,epochs=20,callbacks=early_stop)  

Epoch 1/20


I0000 00:00:1790180319.073592  141572 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_108948__.162


246/530 ━━━━━━━━━━━━━━━━━━━━ 44s 157ms/step - accuracy: 0.6755 - loss: 0.5882

I0000 00:00:1790180362.469222  141572 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_108948__.162


530/530 ━━━━━━━━━━━━━━━━━━━━ 109s 190ms/step - accuracy: 0.7648 - loss: 0.4858 - val_accuracy: 0.7648 - val_loss: 0.4748
Epoch 2/20
530/530 ━━━━━━━━━━━━━━━━━━━━ 84s 158ms/step - accuracy: 0.8221 - loss: 0.4055 - val_accuracy: 0.7667 - val_loss: 0.4564
Epoch 3/20
530/530 ━━━━━━━━━━━━━━━━━━━━ 92s 173ms/step - accuracy: 0.8389 - loss: 0.3819 - val_accuracy: 0.7687 - val_loss: 0.4849
Epoch 4/20
530/530 ━━━━━━━━━━━━━━━━━━━━ 83s 156ms/step - accuracy: 0.8467 - loss: 0.3633 - val_accuracy: 0.7628 - val_loss: 0.4932
Epoch 5/20
530/530 ━━━━━━━━━━━━━━━━━━━━ 86s 163ms/step - accuracy: 0.8530 - loss: 0.3511 - val_accuracy: 0.7874 - val_loss: 0.4493
Epoch 6/20
530/530 ━━━━━━━━━━━━━━━━━━━━ 92s 174ms/step - accuracy: 0.8608 - loss: 0.3414 - val_accuracy: 0.8071 - val_loss: 0.4203
Epoch 7/20
530/530 ━━━━━━━━━━━━━━━━━━━━ 81s 152ms/step - accuracy: 0.8634 - loss: 0.3325 - val_accuracy: 0.8071 - val_loss: 0.4205
Epoch 8/20
530/530 ━━━━━━━━━━━━━━━━━━━━ 82s 154ms/step - accuracy: 0.8631 - loss: 0.3224 - va

In [38]:
res = model.predict(test_set)

122/122 ━━━━━━━━━━━━━━━━━━━━ 16s 105ms/step


In [39]:
y_pred = (res > 0.5).astype(int)

In [40]:
y_true = test_set.classes

In [41]:
print(classification_report(y_true,y_pred))
print(confusion_matrix(y_true,y_pred))

              precision    recall  f1-score   support

           0       0.88      0.72      0.79       951
           1       0.77      0.91      0.83      1000

    accuracy                           0.81      1951
   macro avg       0.83      0.81      0.81      1951
weighted avg       0.82      0.81      0.81      1951

[[681 270]
 [ 92 908]]


In [42]:
model1 = model

In [51]:
base_model.trainable = True

for layer in base_model.layers[:-40]:
    layer.trainable = False


In [52]:
model.compile(optimizer=Adam(learning_rate=1e-5),loss="binary_crossentropy",metrics=["accuracy"])

In [53]:
early_stop = EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True
)
reduce_lr = ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.2,
    patience=2
)

In [54]:
history = model.fit(train_set,validation_data=valid_set,epochs=20,callbacks=[early_stop, reduce_lr],class_weight=class_weight)

Epoch 1/20


I0000 00:00:1790183067.809595  141569 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_230369__.186


332/530 ━━━━━━━━━━━━━━━━━━━━ 37s 188ms/step - accuracy: 0.9143 - loss: 0.2831

I0000 00:00:1790183139.243265  141569 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_230369__.186


530/530 ━━━━━━━━━━━━━━━━━━━━ 137s 229ms/step - accuracy: 0.9144 - loss: 0.2701 - val_accuracy: 0.8632 - val_loss: 0.3064 - learning_rate: 1.0000e-05
Epoch 2/20
530/530 ━━━━━━━━━━━━━━━━━━━━ 91s 171ms/step - accuracy: 0.9128 - loss: 0.2708 - val_accuracy: 0.8583 - val_loss: 0.3176 - learning_rate: 1.0000e-05
Epoch 3/20
530/530 ━━━━━━━━━━━━━━━━━━━━ 95s 179ms/step - accuracy: 0.9172 - loss: 0.2591 - val_accuracy: 0.8760 - val_loss: 0.2857 - learning_rate: 1.0000e-05
Epoch 4/20
530/530 ━━━━━━━━━━━━━━━━━━━━ 106s 199ms/step - accuracy: 0.9190 - loss: 0.2623 - val_accuracy: 0.8691 - val_loss: 0.2936 - learning_rate: 1.0000e-05
Epoch 5/20
530/530 ━━━━━━━━━━━━━━━━━━━━ 88s 166ms/step - accuracy: 0.9270 - loss: 0.2463 - val_accuracy: 0.8750 - val_loss: 0.2851 - learning_rate: 1.0000e-05
Epoch 6/20
530/530 ━━━━━━━━━━━━━━━━━━━━ 92s 173ms/step - accuracy: 0.9272 - loss: 0.2374 - val_accuracy: 0.8652 - val_loss: 0.2965 - learning_rate: 1.0000e-05
Epoch 7/20
530/530 ━━━━━━━━━━━━━━━━━━━━ 106s 199ms/step

In [70]:
resultt = model.predict(test_set)

122/122 ━━━━━━━━━━━━━━━━━━━━ 9s 84ms/step


In [ ]:
result = (resultt > 0.5).astype(int)

In [78]:
print(classification_report(y_true,result))
print(confusion_matrix(y_true,result))

              precision    recall  f1-score   support

           0       0.88      0.89      0.88       951
           1       0.89      0.88      0.89      1000

    accuracy                           0.88      1951
   macro avg       0.88      0.88      0.88      1951
weighted avg       0.88      0.88      0.88      1951

[[845 106]
 [119 881]]


In [79]:
model.save("elbow_x_ray_model.keras")